# SoftRank for Deforestation — *interactive walkthrough*

> Companion notebook to **"Learning to Rank in 2D: Differentiable Spatial Prioritization for Deforestation Forecast"** (Elezi et al., 2026).

This notebook walks step-by-step through the **SoftRank** and **SoftRank+** losses introduced in the paper, applied to a real prediction over the Amazon biome.

By the end you will have built, by hand and in plain `torch`:

1. **Sec. 1** — *what input are we ranking?* Load a saved prediction tensor and the corresponding ground-truth deforestation map.
2. **Sec. 2** — *score → distribution*. Turn each scalar prediction $\hat y_j$ into a Gaussian random variable $s_j \sim \mathcal{N}(\hat y_j, \sigma_s^2)$ and visualize how $\sigma_s$ controls confidence.
3. **Sec. 3** — *who beats whom?* Build the pairwise win-probability matrix $\pi_{ij} = P(s_i > s_j)$.
4. **Sec. 4** — *Tournament Sampling*. Random sampling (Algorithm 1) **vs** Ground-Truth-Injected sampling (Algorithm 2 — *the SoftRank+ contribution*).
5. **Sec. 5** — *rank distributions* via the Rank-Binomial recurrence $p_j^{(i)}(r)$.
6. **Sec. 6** — *SoftNDCG@K* and the **K-truncation trick** (Eq. 12) — why it is the heart of the spatial adaptation.
7. **Sec. 7** — *play with it.* A few "try it yourself" cells: change biweek, change $\sigma_s$, change $K$, change sampler, see the loss move.

The only inputs you need are the two tensors saved in the GitHub release:

    notebooks/data/example_prediction.pt   # (T=24, H, W)
    notebooks/data/example_targets.pt      # (T=24, H, W)

If you trained a model yourself with `python -m scripts.train --loss softrank_plus`, point the cell below to your `results/softrank_plus.pt` and `results/targets.pt` instead — every visualisation in this notebook will adapt automatically.

In [ ]:
from __future__ import annotations
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

# Ensure repo root is on sys.path when running from notebooks/.
import sys
if ".." not in sys.path:
    sys.path.insert(0, "..")

from softrank import config as C
from softrank.losses import _rank_binomial, _soft_ndcg_from_candidates

torch.manual_seed(0)
np.random.seed(0)
plt.rcParams["figure.dpi"] = 110

## 1.  Loading a prediction

We start from a saved score map $\hat y \in [0,1]^{H \times W}$ produced by a trained ResUNet.

* `prediction[t, y, x]` — the model's risk score for cell $(y,x)$ at biweek $t$.
* `target[t, y, x]`     — the ground-truth deforestation intensity at the same cell.

Both tensors are *padded* (the network was trained on padded inputs); we strip the `SIZE`-pixel border to recover the natural Amazon grid before doing anything ranking-related, exactly like the paper does.

In [ ]:
PRED_PATH    = Path("data/example_prediction.pt")   # ← replace with your file
TARGETS_PATH = Path("data/example_targets.pt")

predictions = torch.load(PRED_PATH,    map_location="cpu")[:, C.SIZE:-C.SIZE, C.SIZE:-C.SIZE]
targets     = torch.load(TARGETS_PATH, map_location="cpu")[:, C.SIZE:-C.SIZE, C.SIZE:-C.SIZE]

print("prediction shape:", tuple(predictions.shape), "|  range:",
      f"[{predictions.min():.4f}, {predictions.max():.4f}]")
print("target     shape:", tuple(targets.shape),     "|  range:",
      f"[{targets.min():.4f}, {targets.max():.4f}]")

In [ ]:
# Pick a biweek to inspect throughout the notebook.
BIWEEK = 5     # ← try changing this!
K_BUDGET = 100 # ← operational top-K (paper default).

yhat = predictions[BIWEEK]
y    = targets[BIWEEK]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(yhat, cmap="magma");  axes[0].set_title(f"prediction $\\hat y$ — biweek {BIWEEK}")
axes[1].imshow(y,    cmap="magma");  axes[1].set_title(f"target $g$ — biweek {BIWEEK}")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## 2.  From score to distribution: $\hat y_j \to s_j \sim \mathcal{N}(\hat y_j, \sigma_s^2)$

Standard sorting is non-differentiable. SoftRank fixes this by replacing each scalar $\hat y_j$ with a Gaussian random variable centred on $\hat y_j$:

$$p(s_j) = \mathcal{N}(s_j \mid \hat y_j, \sigma_s^2). \qquad (\text{paper Eq. 6})$$

$\sigma_s$ acts like a **temperature**: small $\sigma_s$ ⇒ confident, almost-hard ranking; large $\sigma_s$ ⇒ soft, fuzzy comparisons that produce stronger gradients early in training.

The cell below shows what changing $\sigma_s$ does to a *single pairwise comparison* between a Top-K cell ($\hat y_i = 0.85$) and a background cell ($\hat y_j = 0.15$).

In [ ]:
from scipy.stats import norm

def pairwise_pi(yi: float, yj: float, sigma_s: float) -> float:
    """Eq. 8: π_{ij} = ½ (1 + erf((yi-yj)/(2 σ_s))) — closed form."""
    return 0.5 * (1 + math.erf((yi - yj) / (2 * sigma_s)))

yi, yj = 0.85, 0.15
for sigma in (0.01, 0.03, 0.1, 0.4):
    print(f"σ_s = {sigma:>4} →  π_ij = {pairwise_pi(yi, yj, sigma):.4f}")

x = np.linspace(yi - yj - 1.0, yi - yj + 1.0, 500)
fig, ax = plt.subplots(figsize=(8, 4))
for sigma, c in zip([0.03, 0.4], ["tab:blue", "tab:red"]):
    pdf = norm.pdf(x, yi - yj, math.sqrt(2) * sigma)
    ax.plot(x, pdf, label=f"σ_s = {sigma}", color=c)
    ax.fill_between(x, pdf, where=x > 0, alpha=0.2, color=c)
ax.axvline(0, color="k", lw=1, ls="-", label="rank boundary")
ax.set_xlabel("$s_i - s_j$  (score difference)")
ax.set_ylabel("density of $s_i-s_j$")
ax.set_title("SoftRank temperature: σ_s controls how sharp the comparison is")
ax.legend(); ax.grid(alpha=.3); plt.show()

## 3.  Tournament Sampling

Computing the full pairwise matrix on a 32×32 patch is *cheap*; doing it on the whole Amazon (~330 000 cells) is intractable. Tournament Sampling picks a tractable subset $\mathcal I_{\text{tournament}}$ of $N$ candidate cells per training step.

The paper compares two sampling strategies:

| | description | paper |
|---|---|---|
| **SoftRank**  (Algorithm 1) | Top-$N_{\text{top}}$ predicted cells **+** uniform random pad | $\mathcal L_{\text{SoftRank}}$ |
| **SoftRank+** (Algorithm 2) | Top-$N_{\text{top}}$ predicted **+** Top-$N_{\text{top}}$ ground-truth (deduped) **+** random pad | $\mathcal L_{\text{SoftRank+}}$ |

The Ground-Truth Injection in SoftRank+ is what fixes paper *Scenario 1* (a true target predicted too low — the model would otherwise never see it). Below we implement both samplers in 15 lines each so you can compare.

In [ ]:
def sample_softrank(prediction, target, Ntop=67, Nrandom=60):
    """Algorithm 1 — random padding only."""
    s = prediction.flatten()
    y = target.flatten()
    _, top_idx = torch.topk(s, k=Ntop)
    available = torch.ones(s.numel(), dtype=torch.bool); available[top_idx] = False
    pool = available.nonzero().flatten()
    rand_idx = pool[torch.randperm(pool.numel())[:Nrandom]]
    cand = torch.cat([top_idx, rand_idx])
    return cand, s[cand], y[cand]

def sample_softrank_plus(prediction, target, Ntop=52, Nrandom=118):
    """Algorithm 2 — Ground-Truth Injection."""
    s = prediction.flatten(); y = target.flatten()
    Ntotal = Ntop + Nrandom
    _, pred_top = torch.topk(s, k=Ntop)
    _, true_top = torch.topk(y, k=Ntop)
    core = torch.cat([pred_top, true_top]).unique()
    need = Ntotal - core.numel()
    available = torch.ones(s.numel(), dtype=torch.bool); available[core] = False
    pool = available.nonzero().flatten()
    fill = pool[torch.randperm(pool.numel())[:need]] if need > 0 else core[:0]
    cand = torch.cat([core, fill]) if need > 0 else core[:Ntotal]
    # sort tournament cells by predicted score for clean π_ij matrices.
    cand = cand[torch.argsort(s[cand], descending=True)]
    return cand, s[cand], y[cand]

cand_a, s_a, y_a = sample_softrank      (yhat, y)
cand_b, s_b, y_b = sample_softrank_plus (yhat, y)
print(f"SoftRank   tournament size: {cand_a.numel()}")
print(f"SoftRank+  tournament size: {cand_b.numel()}")

In [ ]:
def overlay_tournament(prediction, target, cand, title):
    """Show where the sampled cells live on top of the GT map."""
    H, W = prediction.shape
    yy = (cand // W).numpy(); xx = (cand % W).numpy()
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.imshow(target, cmap="magma")
    ax.scatter(xx, yy, s=6, c="cyan", alpha=0.7, label=f"{cand.numel()} sampled")
    ax.set_title(title); ax.legend(loc="lower right"); ax.axis("off")

overlay_tournament(yhat, y, cand_a, "SoftRank — random padding")
overlay_tournament(yhat, y, cand_b, "SoftRank+ — Ground-Truth Injected")
plt.show()

Notice how SoftRank+ places extra blue dots **right on top of bright magma pixels** — that is the Ground-Truth Injection forcing severe deforestation events into every tournament, even when the model has not yet learned to rank them high.

## 4.  The pairwise probability matrix $\pi_{ij}$  (paper Eq. 8 + Fig. 3)

Once we have the sampled scores, we build:

$$\pi_{ij} = P(s_i > s_j) = \tfrac{1}{2}\left[1 + \operatorname{erf}\left(\tfrac{\hat y_i - \hat y_j}{2\sigma_s}\right)\right]$$

In [ ]:
def build_pi_ij(s_candidates, sigma_s):
    diff = s_candidates.unsqueeze(0) - s_candidates.unsqueeze(1)
    sigma_diff = math.sqrt(2) * sigma_s
    return 0.5 * (1 + torch.erf(diff / (sigma_diff * math.sqrt(2))))

pi_ij = build_pi_ij(s_b, sigma_s=C.SOFTRANK_PLUS_SIGMA_S)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(pi_ij.numpy(), cmap="jet", vmin=0, vmax=1)
ax.axhline(C.SOFTRANK_PLUS_NTOP, color="k", lw=1)
ax.axvline(C.SOFTRANK_PLUS_NTOP, color="k", lw=1)
ax.set_title(f"π_ij — SoftRank+ tournament  (N={pi_ij.shape[0]})")
ax.set_xlabel("j"); ax.set_ylabel("i")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=r"π_ij = P(s_i > s_j)")
plt.show()

Interpretation (matches paper Fig. 3):

* **dark red, top-left**: high-confidence wins by the Top-$N_\text{top}$ predicted/GT cells.
* **dark blue, bottom-right**: high-confidence losses by the random-padding cells.
* **fuzzy mid-tone, lower-right block**: ranking ambiguity in the deep background — exactly what the K-truncation in Section 6 below will *zero out* of the gradient.

## 5.  Rank distributions $p_j(r)$ via the Rank-Binomial recurrence

The recurrence in paper Eq. 9:

$$p_j^{(i)}(r) = p_j^{(i-1)}(r-1)\,\pi_{ij} + p_j^{(i-1)}(r)\,(1-\pi_{ij})$$

starts each candidate at rank 0 and "pushes" it down by one whenever it loses a pairwise contest. We re-use the exact implementation from `softrank.losses._rank_binomial` so what you visualize below is what the loss optimises.

In [ ]:
rank_dist = _rank_binomial(pi_ij.unsqueeze(0)).squeeze(0)   # (N, N)

fig, ax = plt.subplots(figsize=(11, 5))
for idx, label, color in zip(
        [0, 1, 2, -1],
        ["Top-1 cell (predicted)", "Top-2", "Top-3", "Background cell"],
        ["#d62728", "#2ca02c", "#1f77b4", "#7f7f7f"]):
    p = rank_dist[idx].numpy()
    ax.step(range(p.size), p, where="mid", label=label, color=color, lw=2)
    ax.fill_between(range(p.size), p, step="mid", alpha=0.1, color=color)
ax.set_xlabel("rank r  (0 = top priority)")
ax.set_ylabel(r"$p_j(r)$")
ax.set_title("Rank distributions for the three Top cells and a background cell")
ax.legend(); ax.grid(alpha=.3); plt.show()

Compare with **Fig. 1** of the paper — same shape: sharp narrow distributions for the highest-scoring cells, broad flat distribution for the background.

Now look at what happens if you squint at the background distribution: most of its mass is at ranks $r \gg K$. **Updating the network's weights to move that mass around does not change Priority@K or NDCG@K** — it is wasted gradient. That is the entire motivation for the K-truncation.

## 6.  SoftNDCG@K and the **K-truncation** (paper Eq. 12 — *the heart of the spatial adaptation*)

$$\mathbf{D}_K = \bigl[D(0),\,D(1),\,\dots,\,D(K-1),\,0,\,\dots,\,0\bigr]^\top \quad\text{with}\quad D(r) = \frac{1}{\log_2(2+r)}.$$

The expected discount is $\mathbb E[D(r_j)]_K = \mathbf{D}_K^\top \mathbf p_j$, and SoftNDCG@K follows the standard NDCG normalisation. **Setting the discounts to zero past rank K is what makes the spatial gradient sparse and operationally meaningful.**

We re-use `softrank.losses._soft_ndcg_from_candidates` so what you see is identical to what gets optimized during training.

In [ ]:
def softrank_loss(prediction, target, sigma_s, K, sampler):
    cand, s_cand, y_cand = sampler(prediction, target)
    s = prediction.flatten().unsqueeze(0)
    y = target.flatten().unsqueeze(0)
    soft_ndcg = _soft_ndcg_from_candidates(
        s_cand.unsqueeze(0), y_cand.unsqueeze(0), y, sigma_s, K)
    return 1.0 - soft_ndcg.item(), soft_ndcg.item()

for label, sampler, sigma in [
    ("SoftRank ", sample_softrank,      C.SOFTRANK_SIGMA_S),
    ("SoftRank+", sample_softrank_plus, C.SOFTRANK_PLUS_SIGMA_S),
]:
    loss, ndcg = softrank_loss(yhat, y, sigma, K_BUDGET, sampler)
    print(f"{label}    SoftNDCG@{K_BUDGET} = {ndcg:.4f}    loss = {loss:.4f}")

## 7.  Try it yourself

Three knobs to play with — change them and rerun this cell:

* **`BIWEEK`** — pick another biweek of 2025 (0–23).
* **`K_TRY`** — operational budget. Smaller K ⇒ stricter shortlist.
* **`SIGMA_TRY`** — temperature.

Watch how SoftRank+ generally beats SoftRank, and how the loss reacts when the temperature gets too high (everyone tied → no gradient) or too low (only the very top of the matrix matters → vanishing signal for the harder cells).

In [ ]:
BIWEEK_TRY = 5
K_TRY      = 100
SIGMA_TRY  = 0.02

yhat_try = predictions[BIWEEK_TRY]
y_try    = targets[BIWEEK_TRY]

print(f"biweek={BIWEEK_TRY}  K={K_TRY}  σ_s={SIGMA_TRY}\n")
for label, sampler in [("SoftRank ", sample_softrank), ("SoftRank+", sample_softrank_plus)]:
    loss, ndcg = softrank_loss(yhat_try, y_try, SIGMA_TRY, K_TRY, sampler)
    print(f"  {label}    SoftNDCG@{K_TRY} = {ndcg:.4f}    loss = {loss:.4f}")

## 8.  Going further

* The full Table I of the paper is reproduced by `python -m scripts.run_experiments`.
* The exact loss code lives in `softrank/losses.py` (≈ 80 lines for SoftRank+).
* The proposed **Tournament Sampling** template generalises to any 2D dense-prediction problem where you only act on the Top-K cells: medical-image triage, disaster response, precision agriculture, etc.

If you build something on top of this, please cite:

```bibtex
@article{elezi2026softrank,
  title   = {Learning to Rank in 2D: Differentiable Spatial Prioritization for Deforestation Forecast},
  author  = {Elezi, Kevin and Feitosa, Raul Queiroz and Ferrari, Felipe and Garza, Paolo and de Souza, Rodrigo Antônio and Bezerra, Francisco Gilney Silva},
  year    = {2026}
}
```